In [ ]:
STAGE = "a"   # "a": nested dead-ball search; "bc": possession rules, then the change detector
import glob, os, subprocess, sys
src = os.path.dirname(glob.glob("/kaggle/input/**/fixed.json", recursive=True)[0])
uv = [sys.executable, "-m", "uv"]
open("/tmp/uv.txt", "w").write('uv==0.5.11 --hash=sha256:08c660c69e7dd874b52ad96b597b57e9f999659f3d9827cdbad884a68e48f7e9\n')
open("/tmp/req.txt", "w").write('numpy==1.26.3 --hash=sha256:5f24750ef94d56ce6e33e4019a8a4d68cfdb1ef661a52cdaee628a56d2437419\npandas==2.1.4 --hash=sha256:e9f17f2b6fc076b2a0078862547595d66244db0f41bf79fc5f64a5c4d635bead\npython-dateutil==4.0.0 --hash=sha256:a5e246f79eb80475e691bf93862cead98decd8a498cf53ba0516fcdc0f50dc1c\npytz==2026.6 --hash=sha256:7fed3fee8fd6bd4a2b7b75a8d56021d0624127bf69bd5691f115db929ac570b3\ntzdata==2026.6 --hash=sha256:0a10eec6e13ef16b627fabed9a5f2ba61a62b1c67f327305f4e7e412a70b4aec\n')
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--require-hashes", "--only-binary", ":all:", "-r", "/tmp/uv.txt"], check=True)
subprocess.run(uv + ["venv", "--python", "3.12.8", "/tmp/env"], check=True)
subprocess.run(uv + ["pip", "install", "--python", "/tmp/env/bin/python", "--require-hashes", "--no-build", "-r", "/tmp/req.txt"], check=True)
subprocess.run(["/tmp/env/bin/python", "-c", "import sys, numpy, pandas; print(sys.version.split()[0], numpy.__version__, pandas.__version__)"], check=True)

In [ ]:
%%writefile /tmp/run.py
"""Runs one stage of the IDSSE searches from the bundle (build.py): stage "a" the nested dead-ball
search (calibrate.run), stage "bc" the possession rules (poss_calibrate.run) and then the change
detector with rule 6 (poss_calibrate.changes_shape_run). Writes aggregates only."""
import glob, json, os, platform, shutil, sys, time
src, work, stage = sys.argv[1], sys.argv[2], sys.argv[3]
pkg = os.path.join(work, "lib")
os.makedirs(pkg, exist_ok=True)
for f in glob.glob(os.path.join(src, "*.py")):
    if os.path.basename(f) != "run.py":
        shutil.copy(f, pkg)
open(os.path.join(pkg, "__init__.py"), "w").close()
os.environ["ARRAYS_DIR"] = src
os.environ["N_WORKERS"] = "4"
sys.path.insert(0, work)
import numpy as np
import pandas as pd
from lib import calibrate as C, deadball, possession, poss_calibrate as P

C.SMOOTH = True
C.WORKERS = 4
cfg = json.load(open(os.path.join(src, "fixed.json")))
deadball.PARAMS.update(cfg["params"])
pr = cfg["profile"]
C.PROFILE = {"bands": np.array(pr["bands"])}
for s in ("dead", "live"):
    C.PROFILE[s] = {"share": pr[s]["share"], "gaps_s": np.array(pr[s]["gaps_s"]), "rate": np.array(pr[s]["rate"])}


def js(x):
    if isinstance(x, dict):
        return {str(k): js(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [js(v) for v in x]
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, np.generic):
        return x.item()
    return x


out = {"versions": [platform.python_version(), np.__version__, pd.__version__], "input": cfg["params"],
       "cpus": os.cpu_count()}
path = os.path.join(work, "out_%s.json" % stage)
t = time.time()
if stage == "a":
    r = C.run()
    r.pop("profile")
    out["a"] = r
    out["a_fixed"] = r["best"] == {k: float(v) for k, v in cfg["params"].items()}
    out["a_s"] = time.time() - t
    json.dump(js(out), open(path, "w"))
    print("a done %.0f s, fixed point reproduced: %s" % (out["a_s"], out["a_fixed"]), flush=True)
else:
    r = P.run()
    out["b"] = r
    out["b_s"] = time.time() - t
    json.dump(js(out), open(path, "w"))
    print("b done %.0f s" % out["b_s"], flush=True)
    possession.PARAMS.update(r["best"])
    out["c"] = P.changes_shape_run()
    out["c_s"] = time.time() - t
    json.dump(js(out), open(path, "w"))
    print("c done %.0f s" % out["c_s"], flush=True)


In [ ]:
env = dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
subprocess.run(["/tmp/env/bin/python", "-u", "-W", "ignore", "/tmp/run.py", src, "/kaggle/working", STAGE],
               check=True, env=env)
for p in glob.glob("/kaggle/working/*"):
    if not os.path.basename(p).startswith("out_"):
        subprocess.run(["rm", "-rf", p])